## Environment Setup

In [1]:
!pip install -U torch torchvision torchaudio
!pip install -U transformers accelerate peft bitsandbytes datasets qwen-vl-utils sentence-transformers json-repair
!pip install tqdm nltk rouge-score bert-score pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.5/915.5 MB 103.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 136.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 111.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 78.2 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 134.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 104.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 145.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 148.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 122.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288

## Imports, Configs & Print Utilities

In [8]:
import os
import re
import json
import collections
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from qwen_vl_utils import process_vision_info
from sklearn.metrics import accuracy_score, f1_score, classification_report
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score

# Force Hugging Face to download massive weights to the persistent volume
os.environ["HF_HOME"] = "/workspace/huggingface_cache"

# --- Global Configurations ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAFE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32

QWEN_BASE_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
TEST1_PATH = "test_1_wo_spec.json"
TEST2_PATH = "test_2_wo_specs.json"
OUTPUT_CSV_PATH = "basemodel_test1_results.csv"

# --- Utility Function ---
def print_fine_grained(labels, preds, metadata_list, title, col_name):
    print(f"\nFine-grained: performance by {title}")
    print("─" * 50)
    print(f"{'':<35} Accuracy  Count")
    print(f"{col_name:<35}")
    
    stats = collections.defaultdict(lambda: {"correct": 0, "total": 0})
    for y_true, y_pred, meta in zip(labels, preds, metadata_list):
        stats[meta]["total"] += 1
        if y_true == y_pred:
            stats[meta]["correct"] += 1
            
    sorted_stats = sorted(stats.items(), key=lambda x: x[1]["total"], reverse=True)
    for k, v in sorted_stats:
        acc = (v["correct"] / v["total"]) * 100
        print(f"{str(k)[:35]:<35} {acc:>7.1f}% {v['total']:>6}")
    print(f"  Total samples accounted for: {len(labels)} / {len(labels)}")

# Load Base Model

In [2]:
print("Loading Qwen Base Model & Processor...")
qwen_processor = AutoProcessor.from_pretrained(QWEN_BASE_ID)

qwen_model = AutoModelForImageTextToText.from_pretrained(
    QWEN_BASE_ID, 
    device_map="auto", 
    torch_dtype=SAFE_DTYPE, 
    attn_implementation="sdpa"
)
qwen_model.eval()
print("✅ Base model loaded successfully!")

Loading Qwen Base Model & Processor...


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

✅ Base model loaded successfully!


# Full Evaluation Loop (Test 1)

In [11]:
TEST1_PATH = "test_1_wo_spec.json"
TEST2_PATH = "test_2_wo_spec.json"
OUTPUT_CSV_PATH = "basemodel_full_results.csv"

def evaluate_base_model(dataset_path, split_name):
    with open(dataset_path, "r", encoding="utf-8") as f:
        dataset = json.load(f)

    true_labels, pred_labels = [], []
    chart_types, reasoning_types = [], []
    all_true_rationales, all_pred_rationales = [], []
    csv_rows = []

    for idx, item in enumerate(tqdm(dataset, desc=f"Evaluating {split_name}")):
        # --- THE PATH FIX FOR RUNPOD (LINUX) ---
        raw_image_path = item.get("local_image_path", "")
        image_path = raw_image_path.replace("\\", "/") # Convert Windows backslashes to Linux forward slashes
        
        claim = item.get("claim", "")
        description = item.get("title_description", "")
        
        # Extract Ground Truths
        true_val_str = "SUPPORTED" if item.get("label") == True else "REFUTED"
        true_val = 1 if true_val_str == "SUPPORTED" else 0
        true_rationale = str(item.get("explanation", "")).strip()

        raw_chart_type = str(item.get("chart_type", "")).strip().lower()
        raw_reasoning = item.get("reasoning_type", "unknown")
        if isinstance(raw_reasoning, list):
            reasoning_str = ", ".join([str(x) for x in raw_reasoning])
        else:
            reasoning_str = str(raw_reasoning)
            
        # Safety skip if image is missing (Now with a debug print!)
        if not os.path.exists(image_path):
            print(f"\n⚠️ Image not found at: '{image_path}'. Skipping sample.")
            continue

        # Format the Zero-Shot Vision Prompt
        prompt_text = f"Claim: {claim}"
        if description:
            prompt_text = f"Chart Description: {description}\n{prompt_text}"

        ver_messages = [
            {
                "role": "system",
                "content": (
                    "You are an expert chart fact-checker. "
                    "You will be given a chart image, its description, and a claim about it. "
                    "Carefully read the chart values and respond in this exact format:\n"
                    "Rationale: <one concise sentence citing the specific chart values>\n"
                    "Verdict: <SUPPORTED or REFUTED>"
                )
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image_path, "max_pixels": 768 * 768},
                    {"type": "text",  "text": prompt_text}
                ]
            }
        ]

        ver_text = qwen_processor.apply_chat_template(
            ver_messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(ver_messages)
        
        inputs = qwen_processor(
            text=[ver_text], images=image_inputs, videos=video_inputs,
            padding=True, return_tensors="pt"
        ).to(DEVICE)

        # Generate Output
        with torch.no_grad():
            ver_ids = qwen_model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=False,
                repetition_penalty=1.05,
                pad_token_id=qwen_processor.tokenizer.pad_token_id,
            )

        ver_trimmed = ver_ids[0][len(inputs.input_ids[0]):]
        output_text = qwen_processor.decode(ver_trimmed, skip_special_tokens=True).strip()

        # Parse Verdict
        pred_val = 0
        pred_val_str = "REFUTED"
        match = re.search(r"Verdict:\s*(SUPPORTED|REFUTED)", output_text, re.IGNORECASE)
        if match:
            pred_val_str = match.group(1).upper()
            if pred_val_str == "SUPPORTED": pred_val = 1
        elif "supported" in output_text.lower():
            pred_val_str = "SUPPORTED"
            pred_val = 1

        # Parse Rationale
        if match:
            pred_rationale = output_text[:match.start()].replace("Rationale:", "").strip()
        else:
            pred_rationale = output_text
            
        true_labels.append(true_val)
        pred_labels.append(pred_val)
        chart_types.append(raw_chart_type)
        reasoning_types.append(reasoning_str)
        all_true_rationales.append(true_rationale)
        all_pred_rationales.append(pred_rationale)
        
        csv_rows.append({
            "Split": split_name,
            "Image Path": image_path,
            "Chart Type": raw_chart_type,
            "Reasoning Type": reasoning_str,
            "Claim": claim,
            "True Verdict": true_val_str,
            "Predicted Verdict": pred_val_str,
            "Verdict Match": true_val_str == pred_val_str,
            "True Rationale": true_rationale,
            "Predicted Rationale": pred_rationale
        })

    # Compute split metrics
    acc = accuracy_score(true_labels, pred_labels) * 100
    f1 = f1_score(true_labels, pred_labels, average="macro") * 100

    print(f"\n[{split_name}] Calculating Text Generation Metrics...")
    smoother = SmoothingFunction().method1
    bleu_scores = [
        sentence_bleu([ref.split()], cand.split(), smoothing_function=smoother) if ref else 0.0
        for ref, cand in zip(all_true_rationales, all_pred_rationales)
    ]
    avg_bleu = np.mean(bleu_scores) * 100 if bleu_scores else 0.0

    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_scores = [
        scorer.score(ref, cand)['rougeL'].fmeasure
        for ref, cand in zip(all_true_rationales, all_pred_rationales)
    ]
    avg_rouge_l = np.mean(rouge_scores) * 100 if rouge_scores else 0.0

    _, _, F1_bert = bert_score(all_pred_rationales, all_true_rationales, lang="en", verbose=False)
    avg_bertscore = F1_bert.mean().item() * 100

    print(f"NLG Results -> BLEU: {avg_bleu:.2f} | ROUGE-L: {avg_rouge_l:.2f} | BERTScore: {avg_bertscore:.2f}")

    return acc, f1, true_labels, pred_labels, chart_types, reasoning_types, csv_rows, avg_bleu, avg_rouge_l, avg_bertscore

# Calculate Metrics & Export Data

In [13]:
print("Evaluating Test 1...")
t1_acc, t1_f1, t1_true, t1_pred, t1_charts, t1_reasons, t1_csv, t1_bleu, t1_rouge, t1_bert = evaluate_base_model(TEST1_PATH, "Test 1")

print("Evaluating Test 2...")
t2_acc, t2_f1, t2_true, t2_pred, t2_charts, t2_reasons, t2_csv, t2_bleu, t2_rouge, t2_bert = evaluate_base_model(TEST2_PATH, "Test 2")

# Calculate Global Averages
avg_acc = (t1_acc + t2_acc) / 2
avg_bleu = (t1_bleu + t2_bleu) / 2
avg_rouge = (t1_rouge + t2_rouge) / 2
avg_bert = (t1_bert + t2_bert) / 2

baseline_acc = 73.8
delta = avg_acc - baseline_acc
delta_str = f"+{delta:.1f}% (above)" if delta >= 0 else f"{delta:.1f}% (below)"

# Print the Final Matrix
SEP = "=" * 115
print(f"\n{SEP}")
print("  CHARTCHECK EVALUATION MATRIX — Qwen Base Model (Zero-Shot)")
print(SEP)
print("                         Model    Task  Test1_Acc  Test1_F1  Test2_Acc  Test2_F1  Avg_Acc  BLEU ROUGE-L BERTScore")
print("          DePlot-DeBERTa-class        C       75.0      75.0       72.5      72.5     73.8     -       -         -")
print("  DePlot-FlanT5-finetune-multi        M       65.7      65.7       65.9      65.8     65.8  17.3    46.3      91.5")
print("         MatCha-finetune-multi        M       59.4      59.1       61.1      60.9     60.2  17.1    37.3      67.8")
print("             GPT4V (Zero-Shot)        M       73.8      73.5       72.0      71.3     72.9  10.0    32.3      89.1")
print(f"     Qwen2.5-VL-7B (Zero-Shot)        M       {t1_acc:<4.1f}      {t1_f1:<4.1f}       {t2_acc:<4.1f}      {t2_f1:<4.1f}     {avg_acc:<4.1f}     {avg_bleu:<4.1f}       {avg_rouge:<4.1f}         {avg_bert:<4.1f}")
print(SEP)

print(f"\n  Avg accuracy vs NLI-DeBERTa baseline : {delta_str}\n")

print(f"Test 1  →  Acc: {t1_acc:.1f}%   Macro-F1: {t1_f1:.1f}")
print(f"Test 2  →  Acc: {t2_acc:.1f}%   Macro-F1: {t2_f1:.1f}")
print(f"Average →  Acc: {avg_acc:.1f}%\n")

print("Test 1 — Per-class report:")
print(classification_report(t1_true, t1_pred, target_names=["REFUTED", "SUPPORTED"], digits=2))

print_fine_grained(t1_true, t1_pred, t1_charts, "chart type (Test 1)", "chart_type\n")
print_fine_grained(t1_true, t1_pred, t1_reasons, "reasoning type (Test 1)", "reasoning_type\n")

# Export combined data
all_results = t1_csv + t2_csv
df_export = pd.DataFrame(all_results)
df_export.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8')

print(f"\n💾 Saved side-by-side qualitative comparison to: {OUTPUT_CSV_PATH}")

Evaluating Test 1...


Evaluating Test 1:   0%|          | 0/939 [00:00<?, ?it/s]


[Test 1] Calculating Text Generation Metrics...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

Evaluating Test 2:   0%|          | 0/981 [00:00<?, ?it/s]


[Test 2] Calculating Text Generation Metrics...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


NLG Results -> BLEU: 5.00 | ROUGE-L: 33.14 | BERTScore: 89.53

  CHARTCHECK EVALUATION MATRIX — Qwen Base Model (Zero-Shot)
                         Model    Task  Test1_Acc  Test1_F1  Test2_Acc  Test2_F1  Avg_Acc  BLEU ROUGE-L BERTScore
          DePlot-DeBERTa-class        C       75.0      75.0       72.5      72.5     73.8     -       -         -
  DePlot-FlanT5-finetune-multi        M       65.7      65.7       65.9      65.8     65.8  17.3    46.3      91.5
         MatCha-finetune-multi        M       59.4      59.1       61.1      60.9     60.2  17.1    37.3      67.8
             GPT4V (Zero-Shot)        M       73.8      73.5       72.0      71.3     72.9  10.0    32.3      89.1
     Qwen2.5-VL-7B (Zero-Shot)        M       82.3      82.2       85.0      85.0     83.7     5.1        32.7         89.5

  Avg accuracy vs NLI-DeBERTa baseline : +9.9% (above)

Test 1  →  Acc: 82.3%   Macro-F1: 82.2
Test 2  →  Acc: 85.0%   Macro-F1: 85.0
Average →  Acc: 83.7%

Test 1 — Per-class r

## Fix Chart types

In [15]:
import pandas as pd
import json
import collections

OUTPUT_CSV_PATH = "basemodel_full_results.csv"  # Update if you named it differently
TEST1_PATH = "test_1_w_spec.json"
TEST2_PATH = "test_2_w_spec.json"

print("Loading original JSON datasets to fetch 'topo' metadata...")
def load_dataset_as_dict(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    # Create a lookup dictionary keyed by (image_path, claim)
    mapping = {}
    for item in data:
        raw_path = item.get("local_image_path", "")
        norm_path = raw_path.replace("\\", "/") # Match the Linux path format from the loop
        claim = item.get("claim", "")
        mapping[(norm_path, claim)] = item
    return mapping

data_map = {}
data_map.update(load_dataset_as_dict(TEST1_PATH))
data_map.update(load_dataset_as_dict(TEST2_PATH))

print("Loading results CSV...")
df = pd.read_csv(OUTPUT_CSV_PATH)

_TOPO_MAP = {
    ("bar",  "vertical"):   "barchart_vertical",
    ("bar",  "horizontal"): "barchart_horizontal",
    ("bar",  "stacked"):    "barchart_vertical",
    ("bar",  ""):           "barchart_vertical",
    ("line", ""):           "line_chart",
    ("line", "area"):       "line_chart",
    ("pie",  ""):           "pie_chart",
    ("pie",  "donut"):      "pie_chart",
    ("scatter", ""):        "scatter_plot",
    ("bubble",  ""):        "bubble_chart",
    ("map",     ""):        "map_chart",
}

updated_chart_types = []

for index, row in df.iterrows():
    img_path = row["Image Path"]
    claim = row["Claim"]
    raw_chart_type = str(row["Chart Type"]).strip().lower()
    
    # Apply your fallback logic
    if not raw_chart_type or raw_chart_type == "unknown":
        item = data_map.get((img_path, claim), {})
        raw_spec = item.get("extended_spec", item.get("raw_spec", {}))
        
        topo_type_raw = raw_spec.get("topo", {}).get("type", "") if isinstance(raw_spec, dict) else ""
        topo_sub_raw = raw_spec.get("topo", {}).get("sub", "") if isinstance(raw_spec, dict) else ""
        
        topo_type = str(topo_type_raw).strip().lower()
        topo_sub  = str(topo_sub_raw).strip().lower()
        
        raw_chart_type = _TOPO_MAP.get(
            (topo_type, topo_sub),
            _TOPO_MAP.get((topo_type, ""), topo_type if topo_type else "unknown")
        )
    
    updated_chart_types.append(raw_chart_type)

# Overwrite the column and save the fixed CSV
df["Chart Type"] = updated_chart_types
df.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8')
print("✅ CSV successfully updated with normalized chart types!\n")

# --- Reprint the Fine-Grained Tables ---
def print_fine_grained(labels, preds, metadata_list, title, col_name):
    print(f"\nFine-grained: performance by {title}")
    print("─" * 50)
    print(f"{'':<35} Accuracy  Count")
    print(f"{col_name:<35}")
    
    stats = collections.defaultdict(lambda: {"correct": 0, "total": 0})
    for y_true, y_pred, meta in zip(labels, preds, metadata_list):
        stats[meta]["total"] += 1
        if y_true == y_pred:
            stats[meta]["correct"] += 1
            
    sorted_stats = sorted(stats.items(), key=lambda x: x[1]["total"], reverse=True)
    for k, v in sorted_stats:
        acc = (v["correct"] / v["total"]) * 100
        print(f"{str(k)[:35]:<35} {acc:>7.1f}% {v['total']:>6}")
    print(f"  Total samples accounted for: {len(labels)} / {len(labels)}\n")

# Split Data for Reprinting
t1_df = df[df["Split"] == "Test 1"]
t2_df = df[df["Split"] == "Test 2"]

for split_name, split_df in [("Test 1", t1_df), ("Test 2", t2_df)]:
    if split_df.empty: continue
    
    # Convert string verdicts back to 1 and 0 for the function
    y_true = (split_df["True Verdict"] == "SUPPORTED").astype(int).tolist()
    y_pred = (split_df["Predicted Verdict"] == "SUPPORTED").astype(int).tolist()
    
    c_types = split_df["Chart Type"].tolist()
    r_types = split_df["Reasoning Type"].tolist()
    
    print_fine_grained(y_true, y_pred, c_types, f"chart type ({split_name})", "chart_type")
    print_fine_grained(y_true, y_pred, r_types, f"reasoning type ({split_name})", "reasoning_type")

Loading original JSON datasets to fetch 'topo' metadata...


JSONDecodeError: Expecting property name enclosed in double quotes: line 56412 column 20 (char 2097152)